In [85]:
import pandas as pd
import numpy as np

In [86]:
# Load downloaded files
players = pd.read_csv("players.csv")
appearances = pd.read_csv("appearances.csv")
games = pd.read_csv('games.csv')

print(f"Players: {len(players)} | Appearances: {len(appearances)} | Games: {len(games)}")

Players: 47701 | Appearances: 1885688 | Games: 88807


In [87]:
players['date_of_birth'] = pd.to_datetime(players['date_of_birth'], errors = 'coerce')
today = pd.to_datetime('today')
players['age'] = (2026 - players['date_of_birth'].dt.year)
players['age'].fillna(players['age'].median())
# players['age'] = players['age'].astype(int)
print('Converted successful')

Converted successful


In [88]:
players.columns

Index(['player_id', 'first_name', 'last_name', 'name', 'last_season',
       'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth',
       'country_of_citizenship', 'date_of_birth', 'sub_position', 'position',
       'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name',
       'image_url', 'international_caps', 'international_goals',
       'current_national_team_id', 'url',
       'current_club_domestic_competition_id', 'current_club_name',
       'market_value_in_eur', 'highest_market_value_in_eur', 'age'],
      dtype='str')

In [89]:
appearances.columns

Index(['appearance_id', 'game_id', 'player_id', 'player_club_id',
       'player_current_club_id', 'date', 'player_name', 'competition_id',
       'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played'],
      dtype='str')

In [90]:
games.columns

Index(['game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id',
       'away_club_id', 'home_club_goals', 'away_club_goals',
       'home_club_position', 'away_club_position', 'home_club_manager_name',
       'away_club_manager_name', 'stadium', 'attendance', 'referee', 'url',
       'home_club_formation', 'away_club_formation', 'home_club_name',
       'away_club_name', 'aggregate', 'competition_type'],
      dtype='str')

In [91]:
players['current_club_name'] = players['current_club_name'].fillna("Unknown Club")

players['market_value_in_eur'] = players['market_value_in_eur'].fillna(players['market_value_in_eur'].median())
players['market_value_in_eur'] = players['market_value_in_eur'].astype(int)

In [92]:
player_stats = appearances.groupby("player_id").agg(
    total_matches = ('appearance_id', 'count'),
    total_minutes = ('minutes_played', 'sum'),
    total_goals = ('goals', 'sum'),
    total_assists = ('assists', 'sum'),
    yellow_cards = ('yellow_cards', 'sum'),
    red_cards = ('red_cards', 'sum')
).reset_index()


# Feature engineering

In [93]:
player_stats["total_minutes"] = player_stats["total_minutes"].replace(0, 90) # avoiding dividing by zero

# ration for 90 mins play
player_stats["goals_per90"] = player_stats["total_goals"] / (player_stats["total_minutes"] / 90)
player_stats["assists_per90"] = player_stats["total_assists"] / (player_stats["total_minutes"] / 90)

# frequencies
player_stats["pass_accuracy"] = 75 + (player_stats["total_assists"] / player_stats["total_matches"] * 3)  # Approximation
player_stats["progressive_passes"] = 2 + (player_stats["total_assists"] / player_stats["total_matches"] * 1.5)
player_stats["dribbles_per90"] = 1.2 + (player_stats["total_goals"] / player_stats["total_matches"] * 2)
player_stats["duels_won_pct"] = 50 + (player_stats["total_goals"] / player_stats["total_matches"] * 2)
player_stats["tackles_per90"] = 1.8 - (player_stats["total_goals"] / player_stats["total_matches"] * 0.5)
player_stats["interceptions_per90"] = 1.5 - (player_stats["total_goals"] / player_stats["total_matches"] * 0.4)
player_stats["aerial_duels_won_pct"] = 48 + (player_stats["total_goals"] / player_stats["total_matches"] * 1.8)

# Ratings
player_stats["rating"] = 6.5 + (player_stats["total_goals"] * 0.2) + (player_stats["total_assists"] * 0.15)

# Replace invalid values
player_stats = player_stats.replace([np.inf, -np.inf], np.nan)
player_stats = player_stats.fillna(player_stats.mean(numeric_only=True))

In [94]:
players_clean = players[[
    "player_id", "name", "position", "sub_position", "age",
    "current_club_name", "market_value_in_eur", "image_url",
    "country_of_citizenship"
]]

# Merge everything
final_data = pd.merge(
    players_clean,
    player_stats,
    on="player_id",
    how="inner"
)

In [95]:
teams = pd.read_csv('R:\\H.H.H\\FIFA  WORLD CUP 2026\\2026 data\\Fifa cup standings.csv')

countries = teams['Team']
print(countries.tolist())

# Keeping only players from these teams
final_data = final_data[final_data["country_of_citizenship"].isin(countries)].reset_index(drop=True)


['Spain', 'Germany', 'Portugal', 'Mexico', 'Czech Republic', 'Canada', 'Brazil', 'USA', 'Ecuador', 'Belgium', 'France', 'Argentina', 'England', 'Switzerland', 'Scotland', 'Türkiye', 'Netherlands', 'Tunisia', 'New Zealand', 'Norway', 'Algeria', 'Colombia', 'Panama', 'Morocco', 'Japan', 'IR Iran', 'Uruguay', 'Austria', 'Croatia', 'South Africa', 'South Korea', 'Bosnia and Herzegovina', 'Qatar', 'Haiti', 'Paraguay', 'Australia', 'Curaçao', "Côte d'Ivoire", 'Sweden', 'Egypt', 'Cabo Verde', 'Saudi Arabia', 'Senegal', 'Iraq', 'Jordan', 'Congo DR', 'Uzbekistan', 'Ghana']


In [96]:
final_data = final_data.rename(columns={
    "name": "player_name",
    "current_club_name": "club",
    "market_value_in_eur": "market_value"
})

final_data = final_data[[
    "player_name", "position", "club", "age", "market_value", "image_url",
    "goals_per90", "assists_per90", "pass_accuracy", "progressive_passes",
    "dribbles_per90", "duels_won_pct", "tackles_per90", "interceptions_per90",
    "aerial_duels_won_pct", "rating"
]]

In [97]:
final_data.to_csv("transfermarkt_players_clean.csv", index=False)
print("\n✅ SUCCESS! All features created!")
print(f"Total players: {len(final_data)}")


✅ SUCCESS! All features created!
Total players: 15868


In [98]:
final_data.isnull().sum()

player_name              0
position                 0
club                     0
age                     18
market_value             0
image_url                0
goals_per90              0
assists_per90            0
pass_accuracy            0
progressive_passes       0
dribbles_per90           0
duels_won_pct            0
tackles_per90            0
interceptions_per90      0
aerial_duels_won_pct     0
rating                   0
dtype: int64

In [113]:
age_med = final_data['age'].median()
final_data['age'] = final_data['age'].fillna(age_med).astype(int)
final_data.isnull().sum()

player_name             0
position                0
club                    0
age                     0
market_value            0
image_url               0
goals_per90             0
assists_per90           0
pass_accuracy           0
progressive_passes      0
dribbles_per90          0
duels_won_pct           0
tackles_per90           0
interceptions_per90     0
aerial_duels_won_pct    0
rating                  0
dtype: int64